In [4]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [5]:
load_dotenv()
llm = ChatOpenAI()

In [6]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation : str

In [7]:
def generate_joke(state : JokeState):
    prompt =f'genarate a jock on the topic {state["topic"]}'
    response = llm.invoke(prompt).content
    return {'joke':response}

In [8]:
def generate_explaination(state : JokeState):
    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [10]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explaination',generate_explaination)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explaination')
graph.add_edge('generate_explaination',END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer =checkpointer)


In [15]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the therapist? Because it had too many topping issues!',
 'explanation': 'This joke plays on the double meaning of the word "topping." In the context of pizza, toppings refer to the ingredients added on top of the pizza such as pepperoni, mushrooms, or olives. The joke suggests that the pizza went to therapy because it had too many "topping issues," implying that it had too many emotional or psychological issues related to its toppings. This humorously anthropomorphizes the pizza, attributing human emotions and problems to it in a lighthearted way.'}

In [16]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the therapist? Because it had too many topping issues!', 'explanation': 'This joke plays on the double meaning of the word "topping." In the context of pizza, toppings refer to the ingredients added on top of the pizza such as pepperoni, mushrooms, or olives. The joke suggests that the pizza went to therapy because it had too many "topping issues," implying that it had too many emotional or psychological issues related to its toppings. This humorously anthropomorphizes the pizza, attributing human emotions and problems to it in a lighthearted way.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1280cf-8b38-6d26-8002-b74c19b6e32b'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-25T05:39:20.243127+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1280cf-74b1-68b1-8001-95cde3afef8a'}}, tasks=

In [17]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the therapist? Because it had too many topping issues!', 'explanation': 'This joke plays on the double meaning of the word "topping." In the context of pizza, toppings refer to the ingredients added on top of the pizza such as pepperoni, mushrooms, or olives. The joke suggests that the pizza went to therapy because it had too many "topping issues," implying that it had too many emotional or psychological issues related to its toppings. This humorously anthropomorphizes the pizza, attributing human emotions and problems to it in a lighthearted way.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1280cf-8b38-6d26-8002-b74c19b6e32b'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-25T05:39:20.243127+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1280cf-74b1-68b1-8001-95cde3afef8a'}}, tasks

In [20]:
config2 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the pasta go to the party?\nBecause it heard it was going to be a penne-cake!',
 'explanation': 'This joke is a play on words. "Penne" is a type of pasta, and "party" sounds like "penne-cake" when spoken quickly. The pasta went to the party because it heard it was going to be a "penne-cake," which is a fun and festive play on words combining "penne" pasta with the word "party."'}

In [21]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the pasta go to the party?\nBecause it heard it was going to be a penne-cake!', 'explanation': 'This joke is a play on words. "Penne" is a type of pasta, and "party" sounds like "penne-cake" when spoken quickly. The pasta went to the party because it heard it was going to be a "penne-cake," which is a fun and festive play on words combining "penne" pasta with the word "party."'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1280d8-5607-6197-800a-7c012d349ef3'}}, metadata={'source': 'loop', 'step': 10, 'parents': {}}, created_at='2026-03-25T05:43:16.257218+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1280d8-413b-69b0-8009-8a8123b9b578'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the pasta go to the party?\nBecause it heard it was going to be a penne-cake!', 'explanation': 'This joke is a

In [22]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1280cf-63dd-6525-8000-dc511af1947b"}})

{'topic': 'pizza',
 'joke': "Why did the pizza go to a therapist? Because it had too many toppings and couldn't handle the pressure!",
 'explanation': "This joke plays on the idea of a pizza having toppings representing stress and pressure. In the joke, the pizza goes to a therapist because it has too many toppings, which it can't handle, symbolizing the idea of feeling overwhelmed or under pressure. It is a light-hearted and humorous way to imagine a pizza seeking therapy for the burden of too many toppings."}

In [25]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1280cf-63dd-6525-8000-dc511af1947b"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1280cf-63dd-6525-8000-dc511af1947b'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-03-25T05:39:16.116202+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1280cf-63c1-612f-bfff-13a71c3acfaa'}}, tasks=(PregelTask(id='f92abb6c-83b6-1592-3571-45014f2f754c', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pizza go to the therapist? Because it had too many topping issues!'}),), interrupts=())

In [24]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1280d8-364f-6340-8008-f02c10e2f24e"}})

{'topic': 'pasta',
 'joke': 'Why did the pasta go to therapy? It had too much emotional baggage-lini.',
 'explanation': 'This joke plays on the pun of "emotional baggage-lini" sounding like "pasta linguine". The joke suggests that the pasta had so much emotional baggage that it needed to go to therapy to help deal with it. The play on words is meant to be funny and lighthearted, as pasta obviously cannot have emotions or go to therapy.'}